# Amazon Marketplace Sales Performance Analysis
**Tools:** Python (Pandas, Seaborn, Matplotlib) | Power BI  
**Dataset:** 100,000 rows × 20 columns | Orders from 2020–2024  
**Author:** Soumya Thamke

---

## Project Overview

This project analyses sales performance across Amazon's marketplace using a real-world-style dataset of 100,000 orders.  
The objective is to identify sales trends, category performance, return rate patterns, discount impact, and payment behaviour — and surface actionable business insights.

**Analysis Flow:**
1. Problem Definition
2. Data Loading & Initial Inspection
3. Data Quality Investigation & Cleaning
4. Exploratory Data Analysis (EDA)
5. Business Insights & Conclusions


---
## Step 1 — Problem Definition

**Business Questions we aim to answer:**
- Which product categories drive the most revenue?
- What is the return rate across categories — and where is it highest?
- How do discounts affect order value?
- Which payment methods are most preferred?
- How has revenue trended year over year from 2020–2024?
- Which markets (countries) contribute the most to sales?


---
## Step 2 — Data Loading & Initial Inspection

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('Amazon.csv')

print("Dataset Shape:", df.shape)
print()
print("Columns:", list(df.columns))
print()
df.head()


In [ ]:
# Data types and basic info
df.info()


In [ ]:
# Basic statistical summary of numerical columns
df[['Quantity', 'UnitPrice', 'Discount', 'Tax', 'ShippingCost', 'TotalAmount']].describe().round(2)


---
## Step 3 — Data Quality Investigation & Cleaning

Before any analysis, we investigate data completeness, accuracy, and consistency.


In [ ]:
# ── 3.1 Check for missing values ──────────────────────────────────────────
print("=== NULL VALUE CHECK ===")
null_counts = df.isnull().sum()
print(null_counts)
print()
print(f"Total nulls found: {null_counts.sum()}")


In [ ]:
# ── 3.2 Check for duplicate Order IDs ─────────────────────────────────────
print("=== DUPLICATE ORDER IDS ===")
dupes = df['OrderID'].duplicated().sum()
print(f"Duplicate Order IDs: {dupes}")


In [ ]:
# ── 3.3 Check for zero or negative TotalAmount ────────────────────────────
print("=== ZERO / NEGATIVE ORDER VALUES ===")
zero_orders = (df['TotalAmount'] <= 0).sum()
print(f"Zero or negative TotalAmount records: {zero_orders}")


In [ ]:
# ── 3.4 Check category label consistency ──────────────────────────────────
print("=== CATEGORY LABELS ===")
print(df['Category'].unique())
print(f"\nTotal unique categories: {df['Category'].nunique()}")

# Standardise casing just in case
df['Category'] = df['Category'].str.strip().str.title()
print("\nAfter standardisation:", df['Category'].unique())


In [ ]:
# ── 3.5 Convert OrderDate to datetime ─────────────────────────────────────
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['Year'] = df['OrderDate'].dt.year
df['Month'] = df['OrderDate'].dt.month
df['YearMonth'] = df['OrderDate'].dt.to_period('M')

print("=== DATE RANGE ===")
print(f"Earliest order: {df['OrderDate'].min().date()}")
print(f"Latest order:   {df['OrderDate'].max().date()}")
print(f"Date range: {df['Year'].min()} – {df['Year'].max()}")


In [ ]:
# ── 3.6 Data Quality Summary Log ──────────────────────────────────────────
quality_log = {
    'Check': [
        'Null values',
        'Duplicate Order IDs',
        'Zero/Negative TotalAmount',
        'Inconsistent Category Labels',
        'Date format conversion'
    ],
    'Issue Found': ['None', 'None', 'None', 'Verified consistent', 'Converted to datetime'],
    'Action Taken': ['No action needed', 'No action needed', 'No action needed', 'Stripped & title-cased', 'Extracted Year, Month, YearMonth']
}

quality_df = pd.DataFrame(quality_log)
print("=== DATA QUALITY LOG ===")
print(quality_df.to_string(index=False))
print()
print("✅ Dataset passed all quality checks. Proceeding to analysis.")


---
## Step 4 — Exploratory Data Analysis (EDA)


### 4.1 Revenue by Category

In [ ]:
cat_revenue = df.groupby('Category')['TotalAmount'].sum().sort_values(ascending=False).reset_index()
cat_revenue.columns = ['Category', 'TotalRevenue']
cat_revenue['TotalRevenue_M'] = (cat_revenue['TotalRevenue'] / 1e6).round(2)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=cat_revenue, x='Category', y='TotalRevenue_M', palette='Blues_d')
plt.title('Total Revenue by Category (USD Millions)', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Revenue (USD Millions)')
for p in ax.patches:
    ax.annotate(f'${p.get_height():.2f}M', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart1_revenue_by_category.png', dpi=150)
plt.show()
print(cat_revenue.to_string(index=False))


### 4.2 Return Rate by Category

In [ ]:
return_rate = df.groupby('Category').apply(
    lambda x: (x['OrderStatus'] == 'Returned').sum() / len(x) * 100
).reset_index()
return_rate.columns = ['Category', 'ReturnRate%']
return_rate = return_rate.sort_values('ReturnRate%', ascending=False)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=return_rate, x='Category', y='ReturnRate%', palette='Reds_d')
plt.title('Return Rate by Category (%)', fontsize=14, fontweight='bold')
plt.xlabel('Category')
plt.ylabel('Return Rate (%)')
plt.axhline(y=return_rate['ReturnRate%'].mean(), color='navy', linestyle='--', label=f"Avg: {return_rate['ReturnRate%'].mean():.2f}%")
plt.legend()
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart2_return_rate_by_category.png', dpi=150)
plt.show()
print(return_rate.to_string(index=False))


### 4.3 Year-over-Year Revenue Trend (2020–2024)

In [ ]:
yearly_rev = df.groupby('Year')['TotalAmount'].sum().reset_index()
yearly_rev.columns = ['Year', 'TotalRevenue']
yearly_rev['TotalRevenue_M'] = (yearly_rev['TotalRevenue'] / 1e6).round(2)
yearly_rev['YoY_Change%'] = yearly_rev['TotalRevenue'].pct_change() * 100

plt.figure(figsize=(10, 5))
sns.lineplot(data=yearly_rev, x='Year', y='TotalRevenue_M', marker='o', color='steelblue', linewidth=2.5, markersize=8)
plt.fill_between(yearly_rev['Year'], yearly_rev['TotalRevenue_M'], alpha=0.1, color='steelblue')
plt.title('Year-over-Year Revenue Trend (USD Millions)', fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Revenue (USD Millions)')
for _, row in yearly_rev.iterrows():
    plt.annotate(f"${row['TotalRevenue_M']}M", (row['Year'], row['TotalRevenue_M']),
                 textcoords="offset points", xytext=(0, 10), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('chart3_yearly_revenue_trend.png', dpi=150)
plt.show()
print(yearly_rev[['Year','TotalRevenue_M','YoY_Change%']].round(2).to_string(index=False))


### 4.4 Discount Impact on Average Order Value

In [ ]:
discount_impact = df.groupby('Discount')['TotalAmount'].mean().reset_index()
discount_impact.columns = ['Discount', 'AvgOrderValue']
discount_impact['Discount_Pct'] = (discount_impact['Discount'] * 100).astype(int).astype(str) + '%'

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=discount_impact, x='Discount_Pct', y='AvgOrderValue', palette='coolwarm_r')
plt.title('Impact of Discount on Average Order Value', fontsize=14, fontweight='bold')
plt.xlabel('Discount (%)')
plt.ylabel('Avg Order Value (USD)')
for p in ax.patches:
    ax.annotate(f'${p.get_height():.0f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart4_discount_impact.png', dpi=150)
plt.show()
print(discount_impact[['Discount_Pct','AvgOrderValue']].round(2).to_string(index=False))


### 4.5 Payment Method Distribution

In [ ]:
payment = df['PaymentMethod'].value_counts().reset_index()
payment.columns = ['PaymentMethod', 'Count']
payment['Share%'] = (payment['Count'] / len(df) * 100).round(2)

plt.figure(figsize=(8, 8))
plt.pie(payment['Count'], labels=payment['PaymentMethod'], autopct='%1.1f%%',
        colors=sns.color_palette('Blues_d', len(payment)), startangle=140)
plt.title('Payment Method Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart5_payment_methods.png', dpi=150)
plt.show()
print(payment.to_string(index=False))


### 4.6 Revenue by Country

In [ ]:
country_rev = df.groupby('Country')['TotalAmount'].sum().sort_values(ascending=False).reset_index()
country_rev.columns = ['Country', 'TotalRevenue']
country_rev['Share%'] = (country_rev['TotalRevenue'] / country_rev['TotalRevenue'].sum() * 100).round(2)

plt.figure(figsize=(10, 5))
ax = sns.barplot(data=country_rev, x='Country', y='TotalRevenue', palette='viridis')
plt.title('Revenue by Country', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Total Revenue (USD)')
for p in ax.patches:
    ax.annotate(f'${p.get_height()/1e6:.1f}M', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('chart6_revenue_by_country.png', dpi=150)
plt.show()
print(country_rev.to_string(index=False))


---
## Step 5 — Business Insights & Conclusions

### Key Findings:

**1. Revenue Distribution is Balanced Across Categories**  
Electronics leads revenue at ~$15.58M but all 6 categories are within 3% of each other — suggesting a well-diversified product portfolio with no single category dependency.

**2. Home & Kitchen Has the Highest Return Rate (3.21%)**  
Home & Kitchen and Books have the highest return rates. Electronics has the lowest at 2.82%. This signals a potential product-fit or quality issue in Home & Kitchen worth investigating.

**3. Discounts Significantly Reduce Average Order Value**  
Orders with 0% discount average ~$988 vs $727 at 30% discount — a 26% drop. Heavy discounting is eroding revenue per order. The business should evaluate whether discount-driven volume offsets this margin loss.

**4. Revenue Has Been Stable 2020–2024 (~$18.3M/year)**  
Year-over-year revenue is remarkably consistent with no major growth or decline. This could indicate market saturation or absence of a growth strategy — worth flagging to the business team.

**5. Credit Cards Dominate (35%) — UPI is Growing**  
Credit Card is the most preferred payment method. UPI at 15% signals strong digital payment adoption, particularly relevant for the Indian market segment.

**6. US Drives 70% of Revenue**  
The United States contributes ~$64.7M of the ~$91.8M total revenue. India (15K orders) and Canada (5.8K) are secondary markets with growth potential.

### Recommendations:
- Investigate return drivers in Home & Kitchen — product descriptions, quality, or sizing issues
- Review discount strategy — cap maximum discount at 20% to protect average order value
- Develop India and Canada market growth plans given their existing order volumes
- Introduce seasonal promotions to break the flat YoY revenue trend


In [ ]:
# Final summary stats
print("=== FINAL SUMMARY ===")
print(f"Total Orders Analysed:     {len(df):,}")
print(f"Total Revenue:             ${df['TotalAmount'].sum()/1e6:.2f}M")
print(f"Average Order Value:       ${df['TotalAmount'].mean():.2f}")
print(f"Overall Return Rate:       {(df['OrderStatus']=='Returned').sum()/len(df)*100:.2f}%")
print(f"Overall Cancellation Rate: {(df['OrderStatus']=='Cancelled').sum()/len(df)*100:.2f}%")
print(f"Date Range:                {df['OrderDate'].min().date()} to {df['OrderDate'].max().date()}")
print(f"Countries:                 {df['Country'].nunique()}")
print(f"Categories:                {df['Category'].nunique()}")
print(f"Unique Brands:             {df['Brand'].nunique()}")
print(f"Unique Sellers:            {df['SellerID'].nunique()}")
